# EIDBench-real — Constraint-set F1 (safe = positive)

Mirrors `eid_bench_experiments/results_analysis/result_full_approach.ipynb` for EIDBench-real validation artifacts, aggregated across multiple example datasets and **multiple repeated generation runs** per (dataset, method, model).

Convention (same as `build_confusion_matrices` in `workflow_prismadv/utils/results_analysis.py`):
- `predicted_as_safe = (num_failed_error == 0)`
- `is_safe = (expected_current_outcome == 'safe')` for corrupted bundles, `True` for the clean bundle
- TP = safe & predicted_safe (correctly cleared benign/clean data)
- FN = safe & predicted_unsafe (false alarm on benign/clean data)
- FP = unsafe & predicted_safe (missed a harmful corruption)
- TN = unsafe & predicted_unsafe (correctly detected a harmful corruption)

Each cell of the F1 table is reported as **mean ± std across runs**. If only one run exists for a cell, the std is `0.00`.


In [1]:
from pathlib import Path
import re
import oyaml as yaml
import numpy as np
import pandas as pd

from prismadv.utils import get_project_root

PROJECT_ROOT = get_project_root()
SCRIPT_ID = "project"

DATASET_ORDER = [
    "bank_marketing_analysis",
    "healthy_diet_dashboard",
    "omop_cdm_databricks",
]

# (display label, artifact prefix, model name)
# Display labels encode (method, model). prismadv = full approach; zero_shot / few_shot / swe_agent = baselines.
# The on-disk script is named single_shot.py but its prompt contains no demonstrations,
# so the correct ML term is "zero-shot". We surface zero_shot in the display labels.
# Model order within each block follows eid_bench (model_order in
# eid_bench_experiments/results_analysis/result_full_approach.ipynb):
# gemini-2.5-flash, gpt-4.1, gpt-4o, gpt-5-mini, gemini-2.5-pro, gpt-5.
METHOD_ROWS = [
    ("prismadv [gemini-2.5-flash]",  "prismadv_real_etl",   "gemini-2.5-flash"),
    ("prismadv [gpt-4.1]",           "prismadv_real_etl",   "gpt-4.1"),
    ("prismadv [gpt-5-mini]",        "prismadv_real_etl",   "gpt-5-mini"),
    ("prismadv [gemini-2.5-pro]",    "prismadv_real_etl",   "gemini-2.5-pro"),
    ("prismadv [gpt-5]",             "prismadv_real_etl",   "gpt-5"),
    ("zero_shot [gpt-5-mini]",       "single_shot_real_etl", "gpt-5-mini"),
    ("zero_shot [gpt-5]",            "single_shot_real_etl", "gpt-5"),
    ("few_shot [gpt-5-mini]",        "few_shot_real_etl",   "gpt-5-mini"),
    ("few_shot [gpt-5]",             "few_shot_real_etl",   "gpt-5"),
    ("swe_agent [gemini-2.5-flash]", "swe_agent_real_etl",  "gemini-2.5-flash"),
    ("swe_agent [gpt-5]",            "swe_agent_real_etl",  "gpt-5"),
]
MODEL_ORDER = [row[0] for row in METHOD_ROWS]

# Timestamp suffix on every artifact name, e.g. ...--20260507_175416
TS_RE = re.compile(r"--(\d{8}_\d{6})$")


def constraints_artifact_dir(example_id: str) -> Path:
    return PROJECT_ROOT / "data_processed" / "eid_bench_real" / example_id / "constraints" / SCRIPT_ID


def discover_runs(example_id: str) -> dict[str, list[str]]:
    """Return {llm_label: [stem_run0, stem_run1, ...]} sorted by timestamp.

    Each run gets its own stem because every generation invocation writes a new
    timestamped YAML; the notebook never re-uses or overwrites prior artifacts.
    """
    base = constraints_artifact_dir(example_id)
    stems_by_label: dict[str, list[str]] = {}
    for label, prefix, model_name in METHOD_ROWS:
        candidates = sorted(
            (p.stem for p in base.glob(f"{prefix}--{model_name}--*.yaml")),
            key=lambda s: (TS_RE.search(s).group(1) if TS_RE.search(s) else s, s),
        )
        if candidates:
            stems_by_label[label] = candidates
    return stems_by_label


def example_paths(example_id: str) -> tuple[Path, Path]:
    validation_root = PROJECT_ROOT / "data_processed" / "eid_bench_real" / example_id / "constraints_validation" / SCRIPT_ID
    error_config_dir = PROJECT_ROOT / "benchmarks" / "EIDBench-real" / example_id / "errors"
    return validation_root, error_config_dir


RUNS_BY_DATASET = {dataset: discover_runs(dataset) for dataset in DATASET_ORDER}
{dataset: {label: len(stems) for label, stems in by_label.items()} for dataset, by_label in RUNS_BY_DATASET.items()}


{'bank_marketing_analysis': {'prismadv [gemini-2.5-flash]': 2,
  'prismadv [gpt-4.1]': 2,
  'prismadv [gpt-5-mini]': 2,
  'prismadv [gemini-2.5-pro]': 2,
  'prismadv [gpt-5]': 2,
  'zero_shot [gpt-5-mini]': 2,
  'zero_shot [gpt-5]': 2,
  'few_shot [gpt-5-mini]': 2,
  'few_shot [gpt-5]': 2,
  'swe_agent [gemini-2.5-flash]': 2,
  'swe_agent [gpt-5]': 2},
 'healthy_diet_dashboard': {'prismadv [gemini-2.5-flash]': 2,
  'prismadv [gpt-4.1]': 2,
  'prismadv [gpt-5-mini]': 2,
  'prismadv [gemini-2.5-pro]': 2,
  'prismadv [gpt-5]': 2,
  'zero_shot [gpt-5-mini]': 2,
  'zero_shot [gpt-5]': 2,
  'few_shot [gpt-5-mini]': 2,
  'few_shot [gpt-5]': 2,
  'swe_agent [gemini-2.5-flash]': 2,
  'swe_agent [gpt-5]': 2},
 'omop_cdm_databricks': {'prismadv [gemini-2.5-flash]': 2,
  'prismadv [gpt-4.1]': 2,
  'prismadv [gpt-5-mini]': 2,
  'prismadv [gemini-2.5-pro]': 2,
  'prismadv [gpt-5]': 2,
  'zero_shot [gpt-5-mini]': 2,
  'zero_shot [gpt-5]': 2,
  'few_shot [gpt-5-mini]': 2,
  'few_shot [gpt-5]': 2,
  's

In [2]:
# Load each error config to learn its expected_current_outcome, per dataset.
def load_corruption_outcomes(config_dir: Path) -> dict:
    outcomes = {}
    for path in sorted(config_dir.glob("*.yaml")):
        cfg = yaml.safe_load(path.read_text())
        outcomes[cfg["label"]] = cfg.get("expected_current_outcome", "unsafe")
    return outcomes

outcomes_by_dataset = {
    example_id: load_corruption_outcomes(example_paths(example_id)[1])
    for example_id in DATASET_ORDER
}
{k: len(v) for k, v in outcomes_by_dataset.items()}


{'bank_marketing_analysis': 20,
 'healthy_diet_dashboard': 20,
 'omop_cdm_databricks': 25}

In [3]:
# Walk validation YAMLs and build one row per (dataset, LLM, run_idx, variant, label).
def summarize_validation(path: Path) -> dict:
    data = yaml.safe_load(path.read_text())
    s = data["summary"]
    total = s["passed_warning"] + s["failed_warning"] + s["passed_error"] + s["failed_error"]
    return {
        "num_passed_warning":  s["passed_warning"],
        "num_failed_warning":  s["failed_warning"],
        "num_passed_error":    s["passed_error"],
        "num_failed_error":    s["failed_error"],
        "num_non_compilable":  s["non_compilable"],
        "total_constraints":   total,
        "predicted_as_safe":   (s["failed_error"] == 0),
    }


rows = []
for example_id in DATASET_ORDER:
    validation_root, _ = example_paths(example_id)
    outcomes = outcomes_by_dataset[example_id]
    for llm, stems in RUNS_BY_DATASET[example_id].items():
        for run_idx, stem in enumerate(stems):
            val_filename = f"validation_results__{stem}.yaml"
            # clean variant (always is_safe=True)
            clean_path = validation_root / "clean" / val_filename
            if clean_path.exists():
                rec = summarize_validation(clean_path)
                rec.update({
                    "dataset_name": example_id,
                    "llm_name":     llm,
                    "run_idx":      run_idx,
                    "run_stem":     stem,
                    "variant":      "clean",
                    "label":        "_clean_",
                    "is_safe":      True,
                })
                rows.append(rec)
            # corrupted variants — is_safe driven by error config
            corr_root = validation_root / "corrupted"
            if not corr_root.exists():
                continue
            for label_dir in sorted(corr_root.iterdir()):
                if not label_dir.is_dir():
                    continue
                label = label_dir.name
                vp = label_dir / val_filename
                if not vp.exists():
                    continue
                rec = summarize_validation(vp)
                rec.update({
                    "dataset_name": example_id,
                    "llm_name":     llm,
                    "run_idx":      run_idx,
                    "run_stem":     stem,
                    "variant":      "corrupted",
                    "label":        label,
                    "is_safe":      outcomes.get(label, "unsafe") == "safe",
                })
                rows.append(rec)

df = pd.DataFrame(rows)
df.groupby(["dataset_name", "llm_name", "run_idx"]).size().rename("n_samples").to_frame()


n_samples
dataset_name            llm_name                    run_idx           
bank_marketing_analysis few_shot [gpt-5-mini]       0               21
                                                    1               21
                        few_shot [gpt-5]            0               21
                                                    1               21
                        prismadv [gemini-2.5-flash] 0               21
...                                                                ...
omop_cdm_databricks     swe_agent [gpt-5]           1               26
                        zero_shot [gpt-5-mini]      0               26
                                                    1               26
                        zero_shot [gpt-5]           0               26
                                                    1               26

[66 rows x 1 columns]

In [4]:
# Per-run confusion matrix and metrics — safe = positive class.
def compute_run_metrics(group: pd.DataFrame) -> pd.Series:
    y_true = group["is_safe"].astype(bool)
    y_pred = group["predicted_as_safe"].astype(bool)
    tp = int((y_true & y_pred).sum())
    fn = int((y_true & ~y_pred).sum())
    fp = int((~y_true & y_pred).sum())
    tn = int((~y_true & ~y_pred).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall    = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0
        else np.nan
    )
    return pd.Series({
        "n_samples":                   int(len(group)),
        "average_constraints":         group["total_constraints"].mean(),
        "average_num_non_compilable":  group["num_non_compilable"].mean(),
        "safe,predicted_as_safe":      tp,
        "safe,predicted_as_unsafe":    fn,
        "unsafe,predicted_as_safe":    fp,
        "unsafe,predicted_as_unsafe":  tn,
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
    })


# Stage 1: one row per (dataset, llm, run_idx) holding that single run's confusion matrix.
per_run_metrics = (
    df.groupby(["dataset_name", "llm_name", "run_idx"], sort=False)
      .apply(compute_run_metrics)
      .reset_index()
)


def aggregate_runs(per_run: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Mean ± std across runs within each group. Counts (TP/FN/etc.) are averaged."""
    agg = per_run.groupby(group_cols, sort=False).agg(
        n_runs=("run_idx", "count"),
        n_samples=("n_samples", "first"),
        average_constraints=("average_constraints", "mean"),
        average_num_non_compilable=("average_num_non_compilable", "mean"),
        TP_mean=("safe,predicted_as_safe", "mean"),
        FN_mean=("safe,predicted_as_unsafe", "mean"),
        FP_mean=("unsafe,predicted_as_safe", "mean"),
        TN_mean=("unsafe,predicted_as_unsafe", "mean"),
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
    )
    # `Series.std` returns NaN for n=1; coerce so single-run cells just display 0.
    for col in ("precision_std", "recall_std", "f1_std"):
        agg[col] = agg[col].fillna(0.0)
    return agg


# Stage 2a: per-dataset aggregate (one row per (dataset, llm) with mean/std across runs of that cell).
per_dataset_metrics = aggregate_runs(per_run_metrics, ["dataset_name", "llm_name"])

# Stage 2b: overall. We pair runs across datasets by run_idx and pool the per-run confusion
# matrices across all 3 datasets — that gives one "overall F1" per matched run index — then
# take mean/std of those. This matches how a single run is currently scored in the paper, while
# attributing run-to-run variance to the same source (the LLM call).
per_overall_run = (
    per_run_metrics
    .groupby(["llm_name", "run_idx"], sort=False)
    .agg(
        n_samples=("n_samples", "sum"),
        average_constraints=("average_constraints", "mean"),
        average_num_non_compilable=("average_num_non_compilable", "mean"),
        **{
            "safe,predicted_as_safe":     ("safe,predicted_as_safe",     "sum"),
            "safe,predicted_as_unsafe":   ("safe,predicted_as_unsafe",   "sum"),
            "unsafe,predicted_as_safe":   ("unsafe,predicted_as_safe",   "sum"),
            "unsafe,predicted_as_unsafe": ("unsafe,predicted_as_unsafe", "sum"),
        },
    )
    .reset_index()
)
# Now compute precision/recall/F1 on those pooled counts per (llm, run_idx).
def _pooled_metrics(row: pd.Series) -> pd.Series:
    tp = row["safe,predicted_as_safe"]
    fn = row["safe,predicted_as_unsafe"]
    fp = row["unsafe,predicted_as_safe"]
    tn = row["unsafe,predicted_as_unsafe"]
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall    = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if np.isfinite(precision) and np.isfinite(recall) and (precision + recall) > 0
        else np.nan
    )
    return pd.Series({"precision": precision, "recall": recall, "f1": f1})

per_overall_run = pd.concat([per_overall_run, per_overall_run.apply(_pooled_metrics, axis=1)], axis=1)
overall_metrics = aggregate_runs(per_overall_run, ["llm_name"]).reindex(
    [m for m in MODEL_ORDER if m in per_overall_run["llm_name"].unique()]
)
per_dataset_metrics.head()


n_runs  n_samples  \
dataset_name            llm_name                                         
bank_marketing_analysis prismadv [gemini-2.5-flash]       2       21.0   
                        prismadv [gpt-4.1]                2       21.0   
                        prismadv [gpt-5-mini]             2       21.0   
                        prismadv [gemini-2.5-pro]         2       21.0   
                        prismadv [gpt-5]                  2       21.0   

                                                     average_constraints  \
dataset_name            llm_name                                           
bank_marketing_analysis prismadv [gemini-2.5-flash]                 33.5   
                        prismadv [gpt-4.1]                          51.0   
                        prismadv [gpt-5-mini]                       55.0   
                        prismadv [gemini-2.5-pro]                   33.5   
                        prismadv [gpt-5]                            57.5   

                                                     average_num_non_compilable  \
dataset_name            llm_name                                                  
bank_marketing_analysis prismadv [gemini-2.5-flash]                         0.0   
                        prismadv [gpt-4.1]                                  0.0   
                        prismadv [gpt-5-mini]                               0.0   
                        prismadv [gemini-2.5-pro]                           0.0   
                        prismadv [gpt-5]                                    0.0   

                                                     TP_mean  FN_mean  \
dataset_name            llm_name                                        
bank_marketing_analysis prismadv [gemini-2.5-flash]      4.0      3.0   
                        prismadv [gpt-4.1]               2.0      5.0   
                        prismadv [gpt-5-mini]            3.0      4.0   
                        prismadv [gemini-2.5-pro]        6.0      1.0   
                        prismadv [gpt-5]                 6.5      0.5   

                                                     FP_mean  TN_mean  \
dataset_name            llm_name                                        
bank_marketing_analysis prismadv [gemini-2.5-flash]      4.5      9.5   
                        prismadv [gpt-4.1]               3.0     11.0   
                        prismadv [gpt-5-mini]            3.0     11.0   
                        prismadv [gemini-2.5-pro]        4.5      9.5   
                        prismadv [gpt-5]                 3.0     11.0   

                                                     precision_mean  \
dataset_name            llm_name                                      
bank_marketing_analysis prismadv [gemini-2.5-flash]        0.458333   
                        prismadv [gpt-4.1]                 0.400000   
                        prismadv [gpt-5-mini]              0.485714   
                        prismadv [gemini-2.5-pro]          0.572727   
                        prismadv [gpt-5]                   0.688889   

                                                     precision_std  \
dataset_name            llm_name                                     
bank_marketing_analysis prismadv [gemini-2.5-flash]       0.294628   
                        prismadv [gpt-4.1]                0.000000   
                        prismadv [gpt-5-mini]             0.121218   
                        prismadv [gemini-2.5-pro]         0.038569   
                        prismadv [gpt-5]                  0.125708   

                                                     recall_mean  recall_std  \
dataset_name            llm_name                                               
bank_marketing_analysis prismadv [gemini-2.5-flash]     0.571429    0.404061   
                        prismadv [gpt-4.1]              0.285714    0.000000   
                        prismadv [gpt-5-mini]           0.428571    0.202031   
      

In [5]:
# Pretty-print precision / recall / F1 per dataset as mean ± std (%).
def _pct_with_std(mean: float, std: float) -> str:
    if not np.isfinite(mean):
        return "--"
    return f"{mean * 100:5.2f} \u00b1 {std * 100:4.2f}"


def format_summary(metrics_frame: pd.DataFrame) -> pd.DataFrame:
    summary = pd.DataFrame({
        "TP":             metrics_frame["TP_mean"].round(2),
        "FN":             metrics_frame["FN_mean"].round(2),
        "TN":             metrics_frame["TN_mean"].round(2),
        "FP":             metrics_frame["FP_mean"].round(2),
        "precision":      [_pct_with_std(m, s) for m, s in zip(metrics_frame["precision_mean"], metrics_frame["precision_std"])],
        "recall":         [_pct_with_std(m, s) for m, s in zip(metrics_frame["recall_mean"], metrics_frame["recall_std"])],
        "F1":             [_pct_with_std(m, s) for m, s in zip(metrics_frame["f1_mean"], metrics_frame["f1_std"])],
        "avg_constraints": metrics_frame["average_constraints"].round(1),
        "n_runs":         metrics_frame["n_runs"].astype(int),
        "n":              metrics_frame["n_samples"].astype(int),
    })
    return summary


for dataset in DATASET_ORDER:
    print(f"=== {dataset} ===")
    if dataset not in per_dataset_metrics.index.get_level_values(0):
        print("  (no validation artifacts found)")
        continue
    sub = per_dataset_metrics.loc[dataset]
    sub = sub.reindex([m for m in MODEL_ORDER if m in sub.index])
    print(format_summary(sub).to_string())
    print()


=== bank_marketing_analysis ===
                               TP   FN    TN    FP      precision         recall             F1  avg_constraints  n_runs   n
llm_name                                                                                                                    
prismadv [gemini-2.5-flash]   4.0  3.0   9.5   4.5  45.83 ± 29.46  57.14 ± 40.41  50.83 ± 34.18             33.5       2  21
prismadv [gpt-4.1]            2.0  5.0  11.0   3.0   40.00 ± 0.00   28.57 ± 0.00   33.33 ± 0.00             51.0       2  21
prismadv [gpt-5-mini]         3.0  4.0  11.0   3.0  48.57 ± 12.12  42.86 ± 20.20  45.24 ± 16.84             55.0       2  21
prismadv [gemini-2.5-pro]     6.0  1.0   9.5   4.5   57.27 ± 3.86   85.71 ± 0.00   68.63 ± 2.77             33.5       2  21
prismadv [gpt-5]              6.5  0.5  11.0   3.0  68.89 ± 12.57  92.86 ± 10.10  79.04 ± 11.96             57.5       2  21
zero_shot [gpt-5-mini]        7.0  0.0   1.5  12.5   36.11 ± 3.93  100.00 ± 0.00   53.00 ± 4.

In [6]:
# Overall aggregated metrics (per-run counts pooled across all datasets, then mean/std).
print("=== Overall (all datasets combined, mean \u00b1 std across runs) ===")
format_summary(overall_metrics)


=== Overall (all datasets combined, mean ± std across runs) ===


,TP,FN,TN,FP,precision,recall,F1,avg_constraints,n_runs,n
llm_name,,,,,,,,,,
prismadv [gemini-2.5-flash],18.5,10.5,23.5,15.5,54.65 ± 2.85,63.79 ± 7.31,58.66 ± 1.47,58.7,2,68
prismadv [gpt-4.1],18.0,11.0,29.5,9.5,65.25 ± 5.25,62.07 ± 9.75,63.57 ± 7.63,57.7,2,68
prismadv [gpt-5-mini],22.5,6.5,29.5,9.5,70.33 ± 0.90,77.59 ± 2.44,73.76 ± 0.61,61.7,2,68
prismadv [gemini-2.5-pro],25.0,4.0,25.0,14.0,64.14 ± 2.33,86.21 ± 0.00,73.55 ± 1.53,40.0,2,68
prismadv [gpt-5],26.5,2.5,31.0,8.0,76.81 ± 0.48,91.38 ± 2.44,83.46 ± 1.30,71.8,2,68
zero_shot [gpt-5-mini],27.5,1.5,7.5,31.5,46.61 ± 1.20,94.83 ± 2.44,62.50 ± 1.61,32.3,2,68
zero_shot [gpt-5],12.5,16.5,25.5,13.5,47.93 ± 5.55,43.10 ± 7.31,45.37 ± 6.55,59.3,2,68
few_shot [gpt-5-mini],27.0,2.0,4.0,35.0,43.56 ± 0.99,93.10 ± 0.00,59.35 ± 0.92,23.8,2,68
few_shot [gpt-5],26.5,2.5,12.5,26.5,50.04 ± 1.34,91.38 ± 2.44,64.64 ± 0.51,25.2,2,68


In [7]:
# LaTeX rows formatted as `mean ± std` (percentages).
def latex_row(model: str, r: pd.Series) -> str:
    def pct(mean, std):
        if not np.isfinite(mean):
            return "--"
        return f"{mean * 100:.2f}\\%~$\\pm$~{std * 100:.2f}\\%"
    return (
        f"& \\texttt{{{model}}} "
        f"& {r['average_constraints']:.1f} "
        f"& {r['average_num_non_compilable']:.1f} "
        f"& {r['TP_mean']:.1f} "
        f"& {r['FN_mean']:.1f} "
        f"& {r['TN_mean']:.1f} "
        f"& {r['FP_mean']:.1f} "
        f"& {pct(r['precision_mean'], r['precision_std'])} "
        f"& {pct(r['recall_mean'], r['recall_std'])} "
        f"& {pct(r['f1_mean'], r['f1_std'])} \\\\"
    )


lines: list[str] = []
for dataset in DATASET_ORDER:
    lines.append(f"%% === Dataset: {dataset} ===")
    if dataset not in per_dataset_metrics.index.get_level_values(0):
        lines.append("%% (no validation artifacts found)")
        continue
    sub = per_dataset_metrics.loc[dataset]
    for model in MODEL_ORDER:
        if model not in sub.index:
            continue
        lines.append(latex_row(model, sub.loc[model]))

lines.append("")
lines.append("%% === Overall (all datasets combined) ===")
for model in MODEL_ORDER:
    if model not in overall_metrics.index:
        continue
    lines.append(latex_row(model, overall_metrics.loc[model]))

print("\n".join(lines))


%% === Dataset: bank_marketing_analysis ===
& \texttt{prismadv [gemini-2.5-flash]} & 33.5 & 0.0 & 4.0 & 3.0 & 9.5 & 4.5 & 45.83\%~$\pm$~29.46\% & 57.14\%~$\pm$~40.41\% & 50.83\%~$\pm$~34.18\% \\
& \texttt{prismadv [gpt-4.1]} & 51.0 & 0.0 & 2.0 & 5.0 & 11.0 & 3.0 & 40.00\%~$\pm$~0.00\% & 28.57\%~$\pm$~0.00\% & 33.33\%~$\pm$~0.00\% \\
& \texttt{prismadv [gpt-5-mini]} & 55.0 & 0.0 & 3.0 & 4.0 & 11.0 & 3.0 & 48.57\%~$\pm$~12.12\% & 42.86\%~$\pm$~20.20\% & 45.24\%~$\pm$~16.84\% \\
& \texttt{prismadv [gemini-2.5-pro]} & 33.5 & 0.0 & 6.0 & 1.0 & 9.5 & 4.5 & 57.27\%~$\pm$~3.86\% & 85.71\%~$\pm$~0.00\% & 68.63\%~$\pm$~2.77\% \\
& \texttt{prismadv [gpt-5]} & 57.5 & 0.0 & 6.5 & 0.5 & 11.0 & 3.0 & 68.89\%~$\pm$~12.57\% & 92.86\%~$\pm$~10.10\% & 79.04\%~$\pm$~11.96\% \\
& \texttt{zero_shot [gpt-5-mini]} & 18.0 & 0.0 & 7.0 & 0.0 & 1.5 & 12.5 & 36.11\%~$\pm$~3.93\% & 100.00\%~$\pm$~0.00\% & 53.00\%~$\pm$~4.24\% \\
& \texttt{zero_shot [gpt-5]} & 43.0 & 0.0 & 2.5 & 4.5 & 11.0 & 3.0 & 45.00\%~$\pm$~7.07

In [8]:
# Per-sample breakdown per dataset — handy for spotting which corruption flipped each LLM's prediction.
# With multiple runs, the cell value is the fraction of runs that predicted "safe" for that (label, llm).
for dataset in DATASET_ORDER:
    print(f"=== {dataset} ===")
    sub = df[df["dataset_name"] == dataset]
    if sub.empty:
        print("  (no rows)")
        continue
    models_present = [m for m in MODEL_ORDER if m in sub["llm_name"].unique()]
    pivot = sub.pivot_table(
        index=["variant", "label", "is_safe"],
        columns="llm_name",
        values="predicted_as_safe",
        aggfunc="mean",
    )[models_present].sort_index()
    print(pivot)
    print()


=== bank_marketing_analysis ===
llm_name                                           prismadv [gemini-2.5-flash]  \
variant   label                           is_safe                                
clean     _clean_                         True                             1.0   
corrupted age_human_range_violation       False                            0.0   
          balance_decimal_cents           True                             0.5   
          balance_minor_high_outliers     True                             0.5   
          balance_scaling_drift           True                             0.5   
          binary_yesno_case_drift         False                            0.0   
          campaign_long_outreach          True                             0.0   
          day_of_week_out_of_range        False                            0.5   
          duration_extreme_outliers       False                            1.0   
          education_ordinal_unseen_label  False                   